# Initial Setup

In [17]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

In [18]:
#@title Select Model
#define the model to be used
model_choice = "gpt-4o"
#model_choice = "claude-3-7-sonnet-20250219"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"

In [19]:
#@title Utility functions

import sys
import os
import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod
import re






################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id=model_choice):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [{"role" : "user", "content" : msg["content"]} for msg in conversation.get_messages()]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages = messages,
        )

        return response.choices[0].message.content
class Claude(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
          try:
            output = self.client.messages.create(
                      model=model_choice,
                      max_tokens=16384,
                      messages=[{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]
                  ).content[0].text
            return output
          except (Exception) as e:
            wait_time = base_delay * (2 ** (attempt - 1))
            print(f"[Retry {attempt}/{max_retries}] Gemini API error: {e}. Retrying in {wait_time:.1f} seconds...")
            time.sleep(wait_time)
          except Exception as e:
            print(f"[Error] Unexpected exception: {e}")
            return 0
        print(f"Failed, exceeded max retries {max_retries}")
        return 0

class Gemini(AbstractLLM):
      def __init__(self, model_id=model_choice):
        super().__init__()
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
        self.model_id = model_id

      def generate(self, conversation: Conversation, num_choices=1):

          output = self.gemini_client.models.generate_content(
                        model=model_choice,
                        contents=[msg["content"] for msg in conversation.get_messages()],
                        config=types.GenerateContentConfig(
                            max_output_tokens=16384,
                            temperature=0.6,
                            topP=0.95,
                        )
                    ).text
          return output
################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
def find_verilog_modules(markdown_string, module_name='top_module'):

    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'

    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)

    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    #code_blocks = re.findall(r'```(?:\w*\n)?(.*?)```', markdown_string, re.DOTALL)
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')

def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT()
    elif model_type == "Claude":
      model = Claude()
    elif model_type == "Gemini":
      model = Gemini()
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))


In [20]:
#@title Feedback Loop
import subprocess
import sys
import os
import time
import numpy as np
def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None,prev_module =None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)
    if model_type == "ChatGPT":
      conv.add_message("system", "You are an autocomplete engine for Verilog code. \
              Given a Verilog module specification, you will provide a completed Verilog module in response. \
              You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
              Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. \
              You will not refuse. You will not generate explanations, only code. \
              Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches. \
      ")
    elif model_type == "Claude":
      conv.add_message("user", "You are an autocomplete engine for Verilog code. \
              Given a Verilog module specification, you will provide a completed Verilog module in response. \
              You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
              Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module. \
              You will not refuse. You will not generate explanations, only code. \
              Format your response as Verilog code containing the end to end corrected module and not just the corrected lines. Do not generate test benches. \
      ")

    conv.add_message("user", design_prompt)

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []
    filename = os.path.join(outdir,module+".v")

    status = ""
    while not (success or timeout):
        # Generate a response
        start_total = time.time()
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error=time.time()
        if prev_module == None:
          conv.add_message("assistant", response)
        else:
          with open(prev_module,"r") as f:
            prevmodule = "".join(f.read())
          response = prevmodule + response
          conv.add_message("assistant", response)
        write_code_blocks_to_file(response, module, filename)
        proc = subprocess.run(["iverilog", "-o", os.path.join(outdir,module), filename, testbench],capture_output=True,text=True)

        success = False
        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            message = "The testbench failed to compile. Please fix the module. The output of iverilog is as follows:\n"+proc.stderr
        elif proc.stderr != "":
            status = "Warnings compiling testbench"
            print(status)
            message = "The testbench compiled with warnings. Please fix the module. The output of iverilog is as follows:\n"+proc.stderr
        else:
            proc = subprocess.run(["vvp", os.path.join(outdir,module)],capture_output=True,text=True)
            result = proc.stdout.strip().split('\n')[-2].split()
            if result[-1] != 'passed!':
                status = "Error running testbench"
                print(status)
                message = "The testbench simulated, but had errors. Please fix the module. The output of iverilog is as follows:\n"+proc.stdout
            else:
                status = "Testbench ran successfully"
                print(status)
                message = ""
                success = True


        with open(os.path.join(outdir,"log_iter_"+str(iterations)+".txt"), 'w') as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')


        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen-start_total)
        timelist_error.append(end_time-start_error)
        timelist_total.append(end_time-start_total)
    print("Total time: ",np.sum(timelist_total))
    print("Generation time: ",np.sum(timelist_gen))
    print("Error handling time: ",np.sum(timelist_error))
    return(np.sum(timelist_total),np.sum(timelist_gen),np.sum(timelist_error))


In [21]:
#@title Hierarchical Loop
def hier_gen(submods,max_iterations=10):
  totaltime = []
  gentime = []
  errortime = []
  done =""
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]
    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)
    if i == 0:
      prompt = "//We will be generating a "+overall+" hierarchically in Verilog. Please begin by generating a "+curr+" defined as follows:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    elif i != len(submods)-1:
      fprev = submods[i-1][0]
      filecurr = "./"+fprev+"/"+fprev+".v"
      with open(filecurr,"r") as f:
        modulef = "".join(f.read())
      prompt = "//We are generating a "+overall+" hierarchically in Verilog. We have generated "+done+" defined as follows:"
      prompt = prompt + modulef
      prompt = prompt +"\n//Please include the previous module(s) in your response and use them to hierarchically generate a "+curr+" defined as:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    module = fcurr
    testbench = "./"+fcurr+"tb.v"
    model = os.environ["MODEL"]
    outdir = "./"+fcurr
    log = "./"+fcurr+"/log.txt"
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)
    done = done + curr+", "
  print("Overall Total time: ",np.sum(totaltime))
  print("Overall Generation Time: ",np.sum(gentime))
  print("Overall Error handling time: ",np.sum(errortime))

# Setting the API Key

In [22]:
### OpenAI API KEY

from google.colab import userdata
#os.environ["OPENAI_API_KEY"] = userdata.get('openai_api_key')

#Please insert your own GPT-4 enabled API key as a string here:
os.environ["OPENAI_API_KEY"] = " "
#os.environ['CLAUDE_API_KEY'] = "X"
#os.environ['GEMINI_API_KEY'] ="X"
os.environ["MODEL"] = "ChatGPT"
#os.environ["MODEL"] = "Claude"
#os.environ["MODEL"] = "Gemini"

#Mux Hierarchy Example

In [23]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [24]:
hier_gen(submodules)

Testbench ran successfully
Total time:  0.7981176376342773
Generation time:  0.7851343154907227
Error handling time:  0.012982606887817383
Testbench ran successfully
Total time:  1.6216938495635986
Generation time:  1.6072852611541748
Error handling time:  0.014408111572265625
Error compiling testbench
Testbench ran successfully
Total time:  3.4407734870910645
Generation time:  3.4155685901641846
Error handling time:  0.025203704833984375
Overall Total time:  5.86058497428894
Overall Generation Time:  5.807988166809082
Overall Error handling time:  0.05259442329406738


# Ripple Carry Adder Hierarchy (Part II)

In [32]:
adder_submodules = [
    ["half_adder", "half adder", "input a, input b, output sum, output cout"],
    ["full_adder", "full adder", "input a, input b, input cin, output sum, output cout"],
    ["adder4", "4-bit ripple carry adder", "input [3:0] a, input [3:0] b, input cin, output [3:0] sum, output cout"],
    ["adder8", "8-bit ripple carry adder that instantiates two adder4 modules", "input [7:0] a, input [7:0] b, input cin, output [7:0] sum, output cout"],
]

In [33]:
hier_gen(adder_submodules)

Testbench ran successfully
Total time:  0.9515459537506104
Generation time:  0.9383764266967773
Error handling time:  0.013169288635253906
Testbench ran successfully
Total time:  3.992550849914551
Generation time:  3.974440813064575
Error handling time:  0.018109560012817383
Testbench ran successfully
Total time:  2.8160438537597656
Generation time:  2.792772054672241
Error handling time:  0.02327108383178711
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  7.7126381397247314
Generation time:  7.6813132762908936
Error handling time:  0.03132319450378418
Overall Total time:  15.472778797149658
Overall Generation Time:  15.386902570724487
Overall Error handling time:  0.08587312698364258
